# 03 Training Science — Batch Norm / Layer Norm

**Status:** Complete

**Learning outcome:** Implement BatchNorm and LayerNorm from scratch in NumPy, verify against PyTorch, and understand when to use each — BatchNorm for CNNs/MLPs, LayerNorm for Transformers.

## Why This Matters

Normalisation layers are in every modern architecture. BatchNorm normalises across the batch dimension (making each feature zero-mean, unit-variance), while LayerNorm normalises across features within each sample. Knowing the difference is critical: BatchNorm depends on batch statistics (breaks with batch_size=1), while LayerNorm is batch-independent (preferred for sequence models).

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
print("Libraries loaded.")

Libraries loaded.


## Theory Sketch

| | **BatchNorm** | **LayerNorm** |
|---|---|---|
| Normalise across | Batch dim (axis=0) | Feature dim (axis=1) |
| Stats per | Feature | Sample |
| Depends on batch? | Yes | No |
| Running stats? | Yes (for eval) | No |
| Best for | CNNs, MLPs | Transformers, RNNs |

Both compute: $\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$, then $y = \gamma \hat{x} + \beta$. The only difference is *which axis* $\mu$ and $\sigma^2$ are computed over.

In [ ]:
axis_fig, axis_axes = plt.subplots(1, 2, figsize=(12, 5))

N_vis, D_vis = 5, 4

# --- BatchNorm panel ---
ax = axis_axes[0]
ax.set_title('BatchNorm\n(normalise across batch for each feature)', fontsize=12, fontweight='bold')
# Draw grid
for i in range(N_vis):
    for j in range(D_vis):
        rect = plt.Rectangle((j, N_vis - 1 - i), 1, 1, fill=True,
                              facecolor='lightyellow', edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(j + 0.5, N_vis - 1 - i + 0.5, f'$x_{{{i},{j}}}$',
                ha='center', va='center', fontsize=9)

# Vertical arrows for each feature column
for j in range(D_vis):
    ax.annotate('', xy=(j + 0.5, -0.3), xytext=(j + 0.5, N_vis + 0.3),
                arrowprops=dict(arrowstyle='->', color='crimson', lw=2.5))
    # Shade column
    rect = plt.Rectangle((j, 0), 1, N_vis, fill=True,
                          facecolor='crimson', alpha=0.08, edgecolor='none')
    ax.add_patch(rect)

ax.set_xlim(-0.5, D_vis + 0.5)
ax.set_ylim(-0.8, N_vis + 0.8)
ax.set_xlabel('Features (D)', fontsize=11)
ax.set_ylabel('Samples (N)', fontsize=11)
ax.set_xticks([j + 0.5 for j in range(D_vis)])
ax.set_xticklabels([f'f{j}' for j in range(D_vis)])
ax.set_yticks([N_vis - 1 - i + 0.5 for i in range(N_vis)])
ax.set_yticklabels([f's{i}' for i in range(N_vis)])
ax.set_aspect('equal')
ax.text(D_vis / 2, -0.6, r'$\mu, \sigma^2$ computed per column (across N)',
        ha='center', fontsize=10, color='crimson', fontstyle='italic')

# --- LayerNorm panel ---
ax = axis_axes[1]
ax.set_title('LayerNorm\n(normalise across features for each sample)', fontsize=12, fontweight='bold')
for i in range(N_vis):
    for j in range(D_vis):
        rect = plt.Rectangle((j, N_vis - 1 - i), 1, 1, fill=True,
                              facecolor='lightyellow', edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(j + 0.5, N_vis - 1 - i + 0.5, f'$x_{{{i},{j}}}$',
                ha='center', va='center', fontsize=9)

# Horizontal arrows for each sample row
for i in range(N_vis):
    ax.annotate('', xy=(D_vis + 0.3, N_vis - 1 - i + 0.5), xytext=(-0.3, N_vis - 1 - i + 0.5),
                arrowprops=dict(arrowstyle='->', color='teal', lw=2.5))
    # Shade row
    rect = plt.Rectangle((0, N_vis - 1 - i), D_vis, 1, fill=True,
                          facecolor='teal', alpha=0.08, edgecolor='none')
    ax.add_patch(rect)

ax.set_xlim(-0.8, D_vis + 0.8)
ax.set_ylim(-0.8, N_vis + 0.8)
ax.set_xlabel('Features (D)', fontsize=11)
ax.set_ylabel('Samples (N)', fontsize=11)
ax.set_xticks([j + 0.5 for j in range(D_vis)])
ax.set_xticklabels([f'f{j}' for j in range(D_vis)])
ax.set_yticks([N_vis - 1 - i + 0.5 for i in range(N_vis)])
ax.set_yticklabels([f's{i}' for i in range(N_vis)])
ax.set_aspect('equal')
ax.text(D_vis / 2, -0.6, r'$\mu, \sigma^2$ computed per row (across D)',
        ha='center', fontsize=10, color='teal', fontstyle='italic')

axis_fig.suptitle('Normalisation Axis: BatchNorm vs LayerNorm', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
axis_fig.savefig('../assets/norm_axis_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print("Axis diagram saved to ../assets/norm_axis_diagram.png")

---
**Understanding Batch vs Layer Dimension**

**Plain language:** Imagine a classroom of students who each took four tests. BatchNorm looks at one test at a time and adjusts so that across all students, that test's scores are centred and spread consistently. LayerNorm looks at one student at a time and adjusts so that across all four of their test scores, the values are centred and spread consistently. Same adjustment, different direction.

**Building intuition:** In a data matrix of shape (N, D) where N is batch size and D is the number of features, BatchNorm computes one mean and one variance *per feature* (per column), using all N samples to calculate them. LayerNorm computes one mean and one variance *per sample* (per row), using all D features. This means BatchNorm's output for a given sample depends on the other samples in the batch, while LayerNorm's output is completely independent of other samples. This independence is why LayerNorm is preferred in Transformers and RNNs, where batch composition should not affect predictions.

**Formal statement:** For input $X \in \mathbb{R}^{N \times D}$, BatchNorm computes statistics along axis 0: $\mu_j = \frac{1}{N}\sum_{i=1}^{N} x_{ij}$ and $\sigma_j^2 = \frac{1}{N}\sum_{i=1}^{N}(x_{ij} - \mu_j)^2$ for each feature $j \in \{1,\dots,D\}$. LayerNorm computes statistics along axis 1: $\mu_i = \frac{1}{D}\sum_{j=1}^{D} x_{ij}$ and $\sigma_i^2 = \frac{1}{D}\sum_{j=1}^{D}(x_{ij} - \mu_i)^2$ for each sample $i \in \{1,\dots,N\}$. The diagram above illustrates these axes visually.

---

## From-Scratch Implementation (NumPy)

In [2]:
def batchnorm_np(x, gamma, beta, eps=1e-5):
    """BatchNorm: normalise across batch (axis=0)."""
    mu = x.mean(axis=0)
    var = x.var(axis=0)  # population variance (ddof=0)
    xhat = (x - mu) / np.sqrt(var + eps)
    return gamma * xhat + beta, mu, var

def layernorm_np(x, gamma, beta, eps=1e-5):
    """LayerNorm: normalise across features (axis=1)."""
    mu = x.mean(axis=1, keepdims=True)
    var = x.var(axis=1, keepdims=True)
    xhat = (x - mu) / np.sqrt(var + eps)
    return gamma * xhat + beta

# Test data
N, D = 8, 4
np.random.seed(42)
x = np.random.randn(N, D) * 3 + 5  # shifted and scaled
gamma = np.ones(D)
beta = np.zeros(D)

bn_out, bn_mu, bn_var = batchnorm_np(x, gamma, beta)
ln_out = layernorm_np(x, gamma, beta)

print("BatchNorm output stats (per feature, should be ~0 mean, ~1 var):")
print(f"  mean: {bn_out.mean(axis=0)}")
print(f"  var:  {bn_out.var(axis=0)}")

print("\nLayerNorm output stats (per sample, should be ~0 mean, ~1 var):")
print(f"  mean: {ln_out.mean(axis=1)}")
print(f"  var:  {ln_out.var(axis=1)}")

assert np.allclose(bn_out.mean(axis=0), 0, atol=1e-6), "BN mean failed"
assert np.allclose(bn_out.var(axis=0), 1, atol=1e-1), "BN var failed"
assert np.allclose(ln_out.mean(axis=1), 0, atol=1e-6), "LN mean failed"
print("\nAll assertions passed ✓")

BatchNorm output stats (per feature, should be ~0 mean, ~1 var):
  mean: [ 2.77555756e-17  2.51534904e-16  8.32667268e-17 -2.49800181e-16]
  var:  [0.99999796 0.99999769 0.99999886 0.99999919]

LayerNorm output stats (per sample, should be ~0 mean, ~1 var):
  mean: [-2.77555756e-16  8.32667268e-17  1.94289029e-16  8.32667268e-17
  1.11022302e-16 -5.55111512e-17 -1.11022302e-16  5.55111512e-17]
  var:  [0.99999684 0.99999808 0.99999418 0.99999856 0.99999733 0.99999895
 0.99999685 0.99999894]

All assertions passed ✓


---
**Understanding What "Normalise" Means and Why It Helps**

**Plain language:** Normalising is like converting all temperatures to the same scale before comparing them. If one thermometer reads in Fahrenheit and another in Celsius, the raw numbers are misleading. Normalisation converts every feature to a common scale — centred at zero, spread out by one unit — so the network can compare them fairly and learn faster.

**Building intuition:** Mathematically, normalising means subtracting the mean (so the data is centred at zero) and dividing by the standard deviation (so the spread becomes one). This is called *standardisation* or *z-scoring*. Neural networks benefit because gradient descent works best when features are on similar scales — otherwise, the loss landscape is elongated and optimisation zig-zags. Normalisation layers apply this transform *inside* the network, not just to the input, keeping activations well-behaved at every layer.

**Formal statement:** Given a set of values $\{x_1, \dots, x_m\}$, normalisation computes $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$ where $\mu = \frac{1}{m}\sum_i x_i$, $\sigma^2 = \frac{1}{m}\sum_i (x_i - \mu)^2$, and $\epsilon$ is a small constant for numerical stability. Learnable parameters $\gamma$ (scale) and $\beta$ (shift) are then applied: $y_i = \gamma\hat{x}_i + \beta$. This ensures the output distribution has controlled mean and variance while preserving the network's representational capacity.

---

## Verification Against PyTorch

In [3]:
# Verify against PyTorch
x_pt = torch.tensor(x, dtype=torch.float32)

# BatchNorm
bn = nn.BatchNorm1d(D, affine=False)
bn.train()
bn_pt = bn(x_pt).detach().numpy()
err_bn = np.max(np.abs(bn_out - bn_pt))
print(f"BatchNorm max error vs PyTorch: {err_bn:.2e}")
assert err_bn < 1e-5, f"BN mismatch: {err_bn}"

# LayerNorm
ln = nn.LayerNorm(D, elementwise_affine=False)
ln_pt = ln(x_pt).detach().numpy()
err_ln = np.max(np.abs(ln_out - ln_pt))
print(f"LayerNorm max error vs PyTorch: {err_ln:.2e}")
assert err_ln < 1e-5, f"LN mismatch: {err_ln}"

print("Both match PyTorch ✓")

BatchNorm max error vs PyTorch: 2.46e-07
LayerNorm max error vs PyTorch: 4.91e-07
Both match PyTorch ✓


---
**Understanding Training vs Inference Difference for BatchNorm**

**Plain language:** BatchNorm behaves differently depending on whether the model is learning or making predictions. During learning, it looks at the current batch to decide how to adjust. During prediction, it uses a long-term average it memorised during training. Forgetting to switch modes is like a student who keeps checking their neighbour's answers during the exam — the results will be wrong and you won't get an error message telling you why.

**Building intuition:** During training, BatchNorm computes the mean and variance from the current mini-batch. Simultaneously, it maintains *running averages* (exponential moving averages) of these statistics. At inference time, we call `model.eval()`, which tells BatchNorm to stop computing batch statistics and instead use the stored running averages. This ensures that the output for a single sample is deterministic and does not depend on what other samples happen to be in the batch. Forgetting `model.eval()` is a common silent bug — the model still produces output, but the statistics are computed from whatever batch is passed in, leading to subtly degraded performance.

**Formal statement:** During training, BatchNorm computes $\mu_B = \frac{1}{m}\sum_{i=1}^{m} x_i$ and $\sigma_B^2 = \frac{1}{m}\sum_{i=1}^{m}(x_i - \mu_B)^2$ from the mini-batch, while updating running statistics: $\hat{\mu} \leftarrow (1-\alpha)\hat{\mu} + \alpha\mu_B$ and $\hat{\sigma}^2 \leftarrow (1-\alpha)\hat{\sigma}^2 + \alpha\sigma_B^2$ (where $\alpha$ is the momentum, default 0.1 in PyTorch). At inference (`model.eval()`), normalisation uses $\hat{\mu}$ and $\hat{\sigma}^2$ instead of batch statistics, making the transform a fixed affine function independent of other inputs.

---

## Visualisation: What Each Norm Does to the Data

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original data
ax = axes[0]
im = ax.imshow(x, aspect='auto', cmap='RdBu_r')
ax.set_title('Original'); ax.set_xlabel('Features'); ax.set_ylabel('Samples')
plt.colorbar(im, ax=ax, shrink=0.8)

# BatchNorm: each column normalised
ax = axes[1]
im = ax.imshow(bn_out, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_title('BatchNorm\n(each column ≈ N(0,1))'); ax.set_xlabel('Features')
plt.colorbar(im, ax=ax, shrink=0.8)

# LayerNorm: each row normalised
ax = axes[2]
im = ax.imshow(ln_out, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_title('LayerNorm\n(each row ≈ N(0,1))'); ax.set_xlabel('Features')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('BatchNorm vs LayerNorm: Normalisation Axes', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/bn_vs_ln.png', dpi=100)
plt.show()
print("Visualisation saved.")

Visualisation saved.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_66705/1103680088.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
hist_fig, hist_axes = plt.subplots(2, 2, figsize=(12, 8))

# Row 0: Before and After BatchNorm
hist_axes[0, 0].hist(x.flatten(), bins=25, color='steelblue', edgecolor='black', alpha=0.8)
hist_axes[0, 0].set_title('Before BatchNorm (raw data)')
hist_axes[0, 0].set_xlabel('Value')
hist_axes[0, 0].set_ylabel('Count')
hist_axes[0, 0].axvline(x.flatten().mean(), color='red', linestyle='--', label=f'mean={x.flatten().mean():.2f}')
hist_axes[0, 0].legend()

hist_axes[0, 1].hist(bn_out.flatten(), bins=25, color='coral', edgecolor='black', alpha=0.8)
hist_axes[0, 1].set_title('After BatchNorm')
hist_axes[0, 1].set_xlabel('Value')
hist_axes[0, 1].set_ylabel('Count')
hist_axes[0, 1].axvline(bn_out.flatten().mean(), color='red', linestyle='--', label=f'mean={bn_out.flatten().mean():.2f}')
hist_axes[0, 1].legend()

# Row 1: Before and After LayerNorm
hist_axes[1, 0].hist(x.flatten(), bins=25, color='steelblue', edgecolor='black', alpha=0.8)
hist_axes[1, 0].set_title('Before LayerNorm (raw data)')
hist_axes[1, 0].set_xlabel('Value')
hist_axes[1, 0].set_ylabel('Count')
hist_axes[1, 0].axvline(x.flatten().mean(), color='red', linestyle='--', label=f'mean={x.flatten().mean():.2f}')
hist_axes[1, 0].legend()

hist_axes[1, 1].hist(ln_out.flatten(), bins=25, color='mediumseagreen', edgecolor='black', alpha=0.8)
hist_axes[1, 1].set_title('After LayerNorm')
hist_axes[1, 1].set_xlabel('Value')
hist_axes[1, 1].set_ylabel('Count')
hist_axes[1, 1].axvline(ln_out.flatten().mean(), color='red', linestyle='--', label=f'mean={ln_out.flatten().mean():.2f}')
hist_axes[1, 1].legend()

hist_fig.suptitle('Distribution of All Values Before/After Normalisation', fontweight='bold', fontsize=14)
plt.tight_layout()
hist_fig.savefig('../assets/bn_ln_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Histogram visualisation saved to ../assets/bn_ln_histograms.png")

## Structured Interpretation

1. **Same formula, different axis**: BatchNorm and LayerNorm are mathematically identical except for the normalisation axis. BatchNorm normalises each feature across the batch; LayerNorm normalises each sample across its features.

2. **BatchNorm creates batch dependence**: During training, each sample's output depends on the other samples in the mini-batch (via shared mean/var). This is fine for i.i.d. data but problematic for sequence models where samples in a batch may have different lengths.

3. **LayerNorm is self-contained**: Each sample is normalised independently, making it suitable for variable-length sequences (Transformers) and online/single-sample inference.

4. **Running statistics**: BatchNorm maintains exponential moving averages of mean and variance for use at test time. LayerNorm doesn't need this — it computes stats per-sample at both train and test time.

5. **For MNEMOSYNE**: Our tabular surrogate model will use BatchNorm (standard for MLPs with fixed-size inputs). If we later add a Transformer component for temporal modeling, it will use LayerNorm.